In [ ]:
!unzip calibration.zip
!pip install ultralytics
from ultralytics import YOLO
model = YOLO("best.pt")

In [ ]:
#@title Locate Real World Pixel Differences Helper Function
import math
def return_box_location(cls, boxes):
  class_location = []
  for box in boxes:
    class_id = int(box.cls[0].item())
    class_name = class_names[class_id]
    if class_name == cls:
      class_location.append(box.xywh[0].tolist())
  return class_location

def check_prediction_classes(prediction):
  boxes = prediction.boxes
  boxes_c = return_box_location('C', boxes)
  boxes_1 = return_box_location('1', boxes)
  assert len(boxes_c) == 2, "Incorrect Number of Control Signal ID"
  assert len(boxes_1) == 1, "Incorrect Number of 1 Signal ID"
  return boxes_c, boxes_1

def roi_slope_center(boxes_c, boxes_1):
  c1 = boxes_c[0]
  c2 = boxes_c[1]
  midpt_c = (c1[0] + c2[0]) // 2, (c1[1] + c2[1]) // 2

  dist_c1_1 = np.sqrt((c1[0] - boxes_1[0][0])**2 + (c1[1] - boxes_1[0][1])**2)
  slope_c1_1 = (c1[1] - boxes_1[0][1]) / (c1[0] - boxes_1[0][0])
  dist_c2_1 = np.sqrt((c2[0] - boxes_1[0][0])**2 + (c2[1] - boxes_1[0][1])**2)
  slope_c2_1 = (c2[1] - boxes_1[0][1]) / (c2[0] - boxes_1[0][0])

  if dist_c1_1 < dist_c2_1:
    return midpt_c, slope_c1_1, dist_c1_1, (boxes_1[0][0], boxes_1[0][1])
  else:
    return midpt_c, slope_c2_1, dist_c2_1, (boxes_1[0][0], boxes_1[0][1])


def calculate_real_world_distance(px_distance, peak, rw_distance=2.55):
    cm_per_pixel = rw_distance / px_distance
    return cm_per_pixel * peak



def sample_signal(img, midpoint, target, slope, distance, tolerance = 5, track_width = 30, show = False):
  gray = img[:,:,1]
  h, w = gray.shape

  angle = math.atan(slope)
  vx = math.cos(angle)
  vy = math.sin(angle)

  vx_to_target = target[0] - midpoint[0]
  vy_to_target = target[1] - midpoint[1]
  dot_product = vx * vx_to_target + vy * vy_to_target

  if dot_product < 0:
      angle += math.pi
      vx = math.cos(angle)
      vy = math.sin(angle)


  x_start = midpoint[0] - tolerance* vx
  y_start = midpoint[1] - tolerance * vy

  x_end = midpoint[0] + (distance + tolerance) * vx
  y_end = midpoint[1] + (distance + tolerance) * vy

  num_steps = int(abs(distance) + (2 * tolerance))
  x_path = np.linspace(x_start, x_end, num_steps)
  y_path = np.linspace(y_start, y_end, num_steps)

  nx = -math.sin(angle)
  ny = math.cos(angle)

  half_track = track_width // 2
  offsets = np.arange(-half_track, half_track)
  signal_sum = np.zeros(num_steps)


  for offset in offsets:
        sample_x = np.clip((x_path + offset * nx).astype(int), 0, w - 1)
        sample_y = np.clip((y_path + offset * ny).astype(int), 0, h - 1)
        signal_sum += gray[sample_y, sample_x]

  avg_signal = signal_sum / track_width

  inverted = np.max(avg_signal) - avg_signal
  win = min(21, len(inverted) // 2 * 2 - 1)
  smooth = savgol_filter(inverted, win, 2) if win >= 3 else inverted
  baseline = gaussian_filter(smooth, sigma=20)
  final_signal = np.maximum(smooth - baseline, 0)

  peaks, _ = find_peaks(final_signal, prominence=np.max(final_signal)*0.02, width=4)


  if show:

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax1.plot(x_path, y_path, color='lime', lw=2, label='Path (with tolerance)')
    ax1.scatter(*midpoint, color='magenta', s=100, label='Midpoint C (Ref)', zorder=5)
    ax1.set_title(f"Sampling Path following Slope {slope:.3f}")
    ax1.legend()

      # Bottom Plot: Signal and Peaks
    ax2.plot(final_signal, color='blue', lw=2, label='Cleaned Signal')
    ax2.fill_between(range(len(final_signal)), final_signal, alpha=0.2, color='blue')

    if len(peaks) > 0:
        ax2.plot(peaks, final_signal[peaks], "x", color='red', markersize=12, mew=2, label='Detected Peaks')
        for p in peaks:
            ax2.text(p, final_signal[p], f'{final_signal[p]:.1f}', color='red', weight='bold')

    ax2.set_title("Quantified Intensity with Peak Detection")
    ax2.set_ylabel("Intensity (AU)")
    ax2.set_xlabel("Pixels along path (including 5px start/end buffer)")
    ax2.legend()

    plt.tight_layout()
    plt.show()

  return peaks, final_signal[peaks]

In [ ]:
#@title Locate Real World Pixel Differences
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import math
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks
import json

band_1 = []
band_2 = []
band_3 = []
band_4 = []
band_5 = []
band_6 = []

class_names = model.names
concentrations = range(1,7)
for concentration in concentrations:
  base_dir = f'/content/calibration/{concentration}/'
  print(base_dir)
  files = [base_dir + f for f in os.listdir(path=base_dir) if not f.startswith('.')]
  results = model(files)
  for idx, result in enumerate(results):
    try:
      box_c_locations, box_1_locations = check_prediction_classes(result)
      midpoint, slope, distance, target = roi_slope_center(box_c_locations, box_1_locations)
      img = cv2.imread(files[idx])
      peak_locations, peak_intensities = sample_signal(img, midpoint, target, slope, distance, track_width=15, tolerance = 30, show = False)
      assert len(peak_locations) == 7

      # Pixel Level Values
      control = peak_locations[0]
      p1_location_px = peak_locations[1] - control
      p2_location_px = peak_locations[2] - control
      p3_location_px = peak_locations[3] - control
      p4_location_px = peak_locations[4] - control
      p5_location_px = peak_locations[5] - control
      p6_location_px = peak_locations[6] - control

      # Real World Values
      p1_location = calculate_real_world_distance(distance, p1_location_px)
      p2_location = calculate_real_world_distance(distance, p2_location_px)
      p3_location = calculate_real_world_distance(distance, p3_location_px)
      p4_location = calculate_real_world_distance(distance, p4_location_px)
      p5_location = calculate_real_world_distance(distance, p5_location_px)
      p6_location = calculate_real_world_distance(distance, p6_location_px)

      band_1.append(p1_location)
      band_2.append(p2_location)
      band_3.append(p3_location)
      band_4.append(p4_location)
      band_5.append(p5_location)
      band_6.append(p6_location)

    except AssertionError as error:
      print(f"Assertion bypassed: {error} on {files[idx]}")

In [ ]:
#@title Quantify Signals Using Distance Only (+/- 3D)
# Quantify Signals Using Known Distances Relative to the Control Peak
def sample_signal_using_distance(
    img,
    midpoint,
    target,
    slope,
    distance,
    real_world_peak_locations,
    real_world_peak_std,
    tolerance=5,
    track_width=30,
    show=False
):
    gray = img[:, :, 1]
    h, w = gray.shape

    # Determine sampling direction
    angle = math.atan(slope)
    vx = math.cos(angle)
    vy = math.sin(angle)

    vx_to_target = target[0] - midpoint[0]
    vy_to_target = target[1] - midpoint[1]

    if vx * vx_to_target + vy * vy_to_target < 0:
        angle += math.pi
        vx = math.cos(angle)
        vy = math.sin(angle)

    # Sampling path
    x_start = midpoint[0] - tolerance * vx
    y_start = midpoint[1] - tolerance * vy
    x_end = midpoint[0] + (distance + tolerance) * vx
    y_end = midpoint[1] + (distance + tolerance) * vy

    num_steps = max(1, int(abs(distance) + 2 * tolerance))
    x_path = np.linspace(x_start, x_end, num_steps)
    y_path = np.linspace(y_start, y_end, num_steps)

    # Average intensity across the sampling track
    nx = -math.sin(angle)
    ny = math.cos(angle)
    offsets = np.arange(-(track_width // 2), track_width // 2)

    signal_sum = np.zeros(num_steps)

    for offset in offsets:
        sample_x = np.clip(
            (x_path + offset * nx).astype(int), 0, w - 1
        )
        sample_y = np.clip(
            (y_path + offset * ny).astype(int), 0, h - 1
        )
        signal_sum += gray[sample_y, sample_x]

    avg_signal = signal_sum / track_width

    # Existing signal-processing pipeline
    inverted = np.max(avg_signal) - avg_signal

    win = min(21, len(inverted) // 2 * 2 - 1)
    smooth = (
        savgol_filter(inverted, win, 2)
        if win >= 3 else inverted
    )

    baseline = gaussian_filter(smooth, sigma=20)
    final_signal = np.maximum(smooth - baseline, 0)

    # Band positions and standard deviations in centimeters,
    # measured relative to the CONTROL peak

    if distance <= 0:
        raise ValueError("Calibration distance must be positive.")

    # Convert physical distance to pixels
    cm_per_pixel = 2.55 / distance

    # The initial tolerance means index 0 is before the midpoint.
    # Convert the midpoint's path position to its signal index.
    midpoint_idx = tolerance

    # Find the CONTROL peak on the left side of the signal.
    # Search only the first half to avoid selecting later bands.
    # Find the first peak from the left
    peaks, _ = find_peaks(final_signal, prominence=np.max(final_signal) * 0.05, width=4)

    if len(peaks) == 0:
        raise ValueError("No peaks detected in the signal.")

    control_idx = peaks[0]
    control_intensity = final_signal[control_idx]
    # Expected band positions are relative to the control peak.
    peak_locations = [control_idx]
    peak_intensities = [control_intensity]

    for location_cm, std_cm in zip(
        real_world_peak_locations, real_world_peak_std
    ):
        offset_px = location_cm / cm_per_pixel
        std_px = std_cm / cm_per_pixel

        expected_idx = control_idx + offset_px

        window_start = max(
            0, int(np.floor(expected_idx - 3 * std_px))
        )
        window_end = min(
            len(final_signal) - 1,
            int(np.ceil(expected_idx + 3 * std_px))
        )

        if window_start <= window_end:
            window = final_signal[window_start:window_end + 1]
            local_idx = int(np.argmax(window))
            peak_idx = window_start + local_idx
            peak_intensity = final_signal[peak_idx]
        else:
            peak_idx = int(np.clip(
                round(expected_idx), 0, len(final_signal) - 1
            ))
            peak_intensity = 0.0

        peak_locations.append(peak_idx)
        peak_intensities.append(peak_intensity)

    peak_locations = np.array(peak_locations, dtype=int)
    peak_intensities = np.array(peak_intensities, dtype=float)

    if show:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

        ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax1.plot(x_path, y_path, color='lime', lw=2,
                 label='Sampling Path')
        ax1.scatter(*midpoint, color='magenta', s=100,
                    label='Reference Midpoint', zorder=5)
        ax1.set_title(f"Sampling Path (Slope {slope:.3f})")
        ax1.legend()

        ax2.plot(final_signal, color='blue', lw=2,
                 label='Processed Signal')

        ax2.plot(control_idx, control_intensity, 'rx',
                 markersize=12, mew=2, label='Control Peak')
        ax2.annotate(
            f'Control: {control_intensity:.1f}',
            (control_idx, control_intensity)
        )

        for i, (location_cm, std_cm) in enumerate(
            zip(real_world_peak_locations, real_world_peak_std),
            start=1
        ):
            expected_idx = control_idx + location_cm / cm_per_pixel
            std_px = std_cm / cm_per_pixel
            peak_idx = peak_locations[i]
            intensity = peak_intensities[i]

            ax2.axvspan(
                max(0, expected_idx - 3 * std_px),
                min(len(final_signal) - 1,
                    expected_idx + 3 * std_px),
                alpha=0.15
            )
            ax2.plot(peak_idx, intensity, 'x',
                     markersize=10)
            ax2.annotate(
                f'Band {i}: {intensity:.1f}',
                (peak_idx, intensity),
                fontsize=9
            )

        ax2.set_title("Control-Referenced Band Intensities")
        ax2.set_ylabel("Intensity (AU)")
        ax2.set_xlabel("Pixels along sampling path")
        ax2.legend()
        plt.tight_layout()
        plt.show()

    return peak_locations, peak_intensities

def calculate_signal_ratios_calibration(signal):
  control = signal[0]
  peak_1 = signal[1]
  peak_2 = signal[2]
  peak_3 = signal[3]
  peak_4 = signal[4]
  peak_5 = signal[5]
  peak_6 = signal[6]

  return peak_1/control, peak_2/control, peak_3/control, peak_4/control, peak_5/control, peak_6/control

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import math
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks
ratios_1 = {key: [] for key in range(1, 7)}
ratios_2 = {key: [] for key in range(1, 7)}
ratios_3 = {key: [] for key in range(1, 7)}
ratios_4 = {key: [] for key in range(1, 7)}
ratios_5 = {key: [] for key in range(1, 7)}
ratios_6 = {key: [] for key in range(1, 7)}

# Real World Locations
band_1_mean = np.mean(band_1)
band_1_std = np.std(band_1)

band_2_mean = np.mean(band_2)
band_2_std = np.std(band_2)

band_3_mean = np.mean(band_3)
band_3_std = np.std(band_3)

band_4_mean = np.mean(band_4)
band_4_std = np.std(band_4)

band_5_mean = np.mean(band_5)
band_5_std = np.std(band_5)

band_6_mean = np.mean(band_6)
band_6_std = np.std(band_6)

peak_locations_cm = {key: [] for key in range(0, 7)}
model = YOLO("best.pt")
class_names = model.names
concentrations = range(1,7)
for concentration in concentrations:
  base_dir = f'/content/calibration/{concentration}/'
  print(base_dir)
  files = [base_dir + f for f in os.listdir(path=base_dir) if not f.startswith('.')]
  results = model(files)
  for idx, result in enumerate(results):
    try:
      box_c_locations, box_1_locations = check_prediction_classes(result)
      midpoint, slope, distance, target = roi_slope_center(box_c_locations, box_1_locations)
      img = cv2.imread(files[idx])


      real_world_peak_locations = [band_1_mean, band_2_mean, band_3_mean, band_4_mean, band_5_mean, band_6_mean]
      real_world_peak_std = [band_1_std, band_2_std, band_3_std, band_4_std, band_5_std, band_6_std]

      peak_locations, peak_intensities = sample_signal_using_distance(img, midpoint, target, slope, distance, list(real_world_peak_locations), list(real_world_peak_std), track_width=15, tolerance = 30, show = False)
      # p1, p2, p3, p4, p5, p6 = calculate_signal_ratios_calibration(peak_intensities)

      ratios_1[concentration].append((peak_intensities[0], peak_intensities[1]))
      ratios_2[concentration].append((peak_intensities[0], peak_intensities[2]))
      ratios_3[concentration].append((peak_intensities[0], peak_intensities[3]))
      ratios_4[concentration].append((peak_intensities[0], peak_intensities[4]))
      ratios_5[concentration].append((peak_intensities[0], peak_intensities[5]))
      ratios_6[concentration].append((peak_intensities[0], peak_intensities[6]))
    except AssertionError as error:
      print(f"Assertion bypassed: {error} on {files[idx]}")

In [ ]:
#@title Visualize 1 image
# Process and visualize ONE image

image_path = '/content/patient_samples/IMG_5522.jpg'  # Change this path

# Run YOLO on the selected image
result = model(image_path)[0]

# Detect control boxes and calculate sampling geometry
box_c_locations, box_1_locations = check_prediction_classes(result)
midpoint, slope, distance, target = roi_slope_center(
    box_c_locations, box_1_locations
)

# Load image
img = cv2.imread(image_path)

# Process signal and visualize
peak_locations, peak_intensities = sample_signal_using_distance(
    img,
    midpoint,
    target,
    slope,
    distance,
    list(real_world_peak_locations),
    list(real_world_peak_std),
    track_width=15,
    tolerance=30,
    show=True
)

# Print control and six band intensities
print(f"Image: {image_path}")
print(f"Control intensity: {peak_intensities[0]:.3f}")

for i in range(1, 7):
    print(f"Band {i} intensity: {peak_intensities[i]:.3f}")

# Calculate ratios relative to control
ratios = peak_intensities[1:] / peak_intensities[0]

for i, ratio in enumerate(ratios, start=1):
    print(f"Band {i} / Control ratio: {ratio:.4f}")

In [ ]:

for i in range(1, 7):
    print(f"Band {i} Mean: {globals()[f'band_{i}_mean']}")
    print(f"Band {i} Std:  {globals()[f'band_{i}_std']}")
    print("-" * 30)


In [ ]:
#@title Build dataset to train NN

import pandas as pd
def convert_dict_to_pandas(data):
  df = pd.DataFrame(columns=["Control", "Signal", "Target"])
  for key, values in data.items():
    for (c, s) in values:
      df.loc[len(df)] = [c,s,key]
  return df

df_1 = convert_dict_to_pandas(ratios_1)
df_2 = convert_dict_to_pandas(ratios_2)
df_3 = convert_dict_to_pandas(ratios_3)
df_4 = convert_dict_to_pandas(ratios_4)
df_5 = convert_dict_to_pandas(ratios_5)
df_6 = convert_dict_to_pandas(ratios_6)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import TensorDataset, DataLoader


# =====================================================
# MODEL
# =====================================================

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


# =====================================================
# MAIN FUNCTION
# =====================================================

def train_nn_cv(df, target_col="Target", control_col="Control", signal_col="Signal",
                n_splits=5, epochs=150, batch_size=32, lr=0.001, exclude=[]):

    df_clean = df.copy()
    df_clean = df_clean[~df_clean[target_col].isin(exclude)]

    # -----------------------------
    # DATA
    # -----------------------------
    X = df_clean[[control_col, signal_col]].values
    y = df_clean[target_col].values.reshape(-1, 1)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_results = []
    fold_models = []
    fold_scalers = []

    # -----------------------------
    # CROSS VALIDATION
    # -----------------------------
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"\n===== FOLD {fold+1} =====")

        X_train_raw, X_val_raw = X[train_idx], X[val_idx]
        y_train_raw, y_val_raw = y[train_idx], y[val_idx]

        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        X_train = scaler_X.fit_transform(X_train_raw)
        X_val = scaler_X.transform(X_val_raw)

        y_train = scaler_y.fit_transform(y_train_raw)
        y_val = scaler_y.transform(y_val_raw)

        X_train = torch.FloatTensor(X_train)
        y_train = torch.FloatTensor(y_train)
        X_val = torch.FloatTensor(X_val)
        y_val = torch.FloatTensor(y_val)

        train_loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=batch_size,
            shuffle=True
        )

        model = Net()
        criterion = nn.HuberLoss(delta=1.0)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # TRAIN
        for _ in range(epochs):
            model.train()
            for xb, yb in train_loader:
                optimizer.zero_grad()
                pred = model(xb)
                loss = criterion(pred, yb)
                loss.backward()
                optimizer.step()

        # VALIDATION
        model.eval()
        with torch.no_grad():
            pred_scaled = model(X_val).numpy()

        pred = scaler_y.inverse_transform(pred_scaled).flatten()
        true = y_val_raw.flatten()

        mae = mean_absolute_error(true, pred)
        rmse = np.sqrt(mean_squared_error(true, pred))
        r2 = r2_score(true, pred)

        print(f"MAE : {mae:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"R²  : {r2:.4f}")

        fold_results.append((mae, rmse, r2))
        fold_models.append(model)
        fold_scalers.append((scaler_X, scaler_y))

    # -----------------------------
    # BEST FOLD (for reference)
    # -----------------------------
    best_idx = np.argmin([r[0] for r in fold_results])
    best_model = fold_models[best_idx]
    scaler_X_best, scaler_y_best = fold_scalers[best_idx]

    print(f"\nBest fold: {best_idx + 1}")

    # -----------------------------
    # FULL DATA TRAINING (FINAL MODEL)
    # -----------------------------
    scaler_X_full = StandardScaler()
    scaler_y_full = StandardScaler()

    X_full = scaler_X_full.fit_transform(X)
    y_full = scaler_y_full.fit_transform(y)

    X_full_t = torch.FloatTensor(X_full)
    y_full_t = torch.FloatTensor(y_full)

    full_loader = DataLoader(
        TensorDataset(X_full_t, y_full_t),
        batch_size=batch_size,
        shuffle=True
    )

    final_model = Net()
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.Adam(final_model.parameters(), lr=lr)

    for _ in range(epochs):
        final_model.train()
        for xb, yb in full_loader:
            optimizer.zero_grad()
            pred = final_model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

    # -----------------------------
    # FINAL PREDICTION (for plots)
    # -----------------------------
    final_model.eval()
    with torch.no_grad():
        pred_scaled = final_model(X_full_t).numpy()

    pred = scaler_y_full.inverse_transform(pred_scaled).flatten()
    true = y.flatten()

    ratio = df_clean[signal_col] / df_clean[control_col]

    # =====================================================
    # PLOTS
    # =====================================================

    idx = np.argsort(pred)

    plt.figure(figsize=(10,6))
    plt.scatter(df_clean[target_col], ratio, alpha=0.6, label="Observed")
    plt.plot(pred[idx], ratio.iloc[idx], linewidth=3, label="NN Fit (Final Model)")
    plt.xlabel("Concentration")
    plt.ylabel("Signal / Control")
    plt.title("Concentration vs Ratio")
    plt.legend()
    plt.grid()
    plt.show()

    plt.figure(figsize=(6,6))
    plt.scatter(true, pred, alpha=0.7)
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    plt.plot(lims, lims, '--')
    plt.xlabel("True Concentration")
    plt.ylabel("Predicted Concentration")
    plt.title("Predicted vs Actual")
    plt.grid()
    plt.show()

    residuals = pred - true
    plt.figure(figsize=(8,5))
    plt.scatter(true, residuals, alpha=0.7)
    plt.axhline(0, linestyle='--')
    plt.xlabel("True Concentration")
    plt.ylabel("Residual")
    plt.title("Residual Plot")
    plt.grid()
    plt.show()

    # -----------------------------
    # SUMMARY METRICS
    # -----------------------------
    fold_results = np.array(fold_results)

    print("\n==============================")
    print("        FINAL CV RESULTS       ")
    print("==============================")
    print(f"MAE  : {fold_results[:,0].mean():.4f} ± {fold_results[:,0].std():.4f}")
    print(f"RMSE : {fold_results[:,1].mean():.4f} ± {fold_results[:,1].std():.4f}")
    print(f"R²   : {fold_results[:,2].mean():.4f} ± {fold_results[:,2].std():.4f}")
    print("==============================\n")

    # -----------------------------
    # RETURN EVERYTHING USEFUL
    # -----------------------------
    return {
        "best_fold": best_idx + 1,
        "fold_results": fold_results,
        "best_model": best_model,
        "final_model": final_model,
        "scaler_X": scaler_X_full,
        "scaler_y": scaler_y_full,
        "pred": pred,
        "true": true
    }

In [ ]:
import torch

dfs = [df_1, df_2, df_3, df_4, df_5, df_6]

all_results = []

for i, df in enumerate(dfs, start=1):
    print(f"\n==============================")
    print(f"Training model for df_{i}")
    print(f"==============================")

    results = train_nn_cv(df)

    save_path = f"final_model_band{i}.pth"

    torch.save({
        "model_state_dict": results["final_model"].state_dict(),

        "scaler_X_mean": results["scaler_X"].mean_.tolist(),
        "scaler_X_scale": results["scaler_X"].scale_.tolist(),

        "scaler_y_mean": results["scaler_y"].mean_.tolist(),
        "scaler_y_scale": results["scaler_y"].scale_.tolist(),
    }, save_path)

    print(f"Saved: {save_path}")

    all_results.append({
        "dataset": f"df_{i}",
        "best_fold": results["best_fold"],
        "fold_results": results["fold_results"]
    })

In [ ]:
#@title Patient Samples
!unzip patient_samples.zip
!unzip positive_control.zip
!unzip negative_control.zip
#!pip install ultralytics
from ultralytics import YOLO

In [ ]:

# ============================================================
# STANDALONE PATIENT INFERENCE — NO RETRAINING
# ============================================================

from pathlib import Path
import os
import math
import warnings

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from scipy.signal import savgol_filter, find_peaks
from scipy.ndimage import gaussian_filter
from ultralytics import YOLO


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path("/content")
PATIENT_CSV = BASE_DIR / "patient_readings.csv"
YOLO_PATH = BASE_DIR / "best.pt"

MODEL_PATHS = {
    band: BASE_DIR / f"final_model_band{band}.pth"
    for band in range(1, 7)
}

OUTPUT_DIR = BASE_DIR / "patient_results"
PLOT_DIR = OUTPUT_DIR / "diagnostic_plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")


# ============================================================
# EXACT NEURAL NETWORK ARCHITECTURE FROM TRAINING
# ============================================================

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# LOAD ALL SIX MODELS AND THEIR SAVED SCALERS
# ============================================================

def load_band_model(model_path):
    if not model_path.is_file():
        raise FileNotFoundError(f"Model not found: {model_path}")

    checkpoint = torch.load(
        str(model_path),
        map_location=DEVICE,
        weights_only=True
    )

    net = Net().to(DEVICE)
    net.load_state_dict(checkpoint["model_state_dict"])
    net.eval()

    required_keys = [
        "scaler_X_mean",
        "scaler_X_scale",
        "scaler_y_mean",
        "scaler_y_scale"
    ]

    for key in required_keys:
        if key not in checkpoint:
            raise KeyError(f"{model_path.name} missing {key}")

    scaler_X_mean = np.asarray(
        checkpoint["scaler_X_mean"], dtype=np.float32
    ).reshape(1, -1)

    scaler_X_scale = np.asarray(
        checkpoint["scaler_X_scale"], dtype=np.float32
    ).reshape(1, -1)

    scaler_y_mean = np.asarray(
        checkpoint["scaler_y_mean"], dtype=np.float32
    ).reshape(-1)

    scaler_y_scale = np.asarray(
        checkpoint["scaler_y_scale"], dtype=np.float32
    ).reshape(-1)

    if scaler_X_mean.shape[1] != 2:
        raise ValueError(
            f"{model_path.name}: expected 2 input features, "
            f"got {scaler_X_mean.shape[1]}"
        )

    if np.any(scaler_X_scale == 0):
        raise ValueError(f"Invalid input scaler in {model_path.name}")

    if np.any(scaler_y_scale == 0):
        raise ValueError(f"Invalid output scaler in {model_path.name}")

    return {
        "model": net,
        "X_mean": scaler_X_mean,
        "X_scale": scaler_X_scale,
        "y_mean": scaler_y_mean,
        "y_scale": scaler_y_scale
    }


band_models = {}

for band, path in MODEL_PATHS.items():
    band_models[band] = load_band_model(path)
    print(f"Loaded band {band}: {path.name}")


# ============================================================
# PREDICT ONE BAND
# ============================================================

def predict_band(control_intensity, signal_intensity, model_info):
    """
    Inputs must be in the same units and order as the training
    columns: [Control, Signal].
    """

    X = np.array(
        [[control_intensity, signal_intensity]],
        dtype=np.float32
    )

    X_scaled = (
        X - model_info["X_mean"]
    ) / model_info["X_scale"]

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    with torch.no_grad():
        y_scaled = (
            model_info["model"](X_tensor)
            .cpu()
            .numpy()
        )

    prediction = (
        y_scaled * model_info["y_scale"]
        + model_info["y_mean"]
    )

    return float(prediction.reshape(-1)[0])


# ============================================================
# IMAGE LOOKUP
# ============================================================

def find_image(image_name):
    image_name = str(image_name).strip()

    if not image_name or image_name.lower() == "nan":
        return None

    candidate = Path(image_name)

    if candidate.is_absolute() and candidate.is_file():
        return candidate

    direct_path = BASE_DIR / candidate

    if direct_path.is_file():
        return direct_path

    matches = [
        p for p in BASE_DIR.rglob(candidate.name)
        if p.is_file()
        and OUTPUT_DIR not in p.parents
    ]

    if matches:
        if len(matches) > 1:
            print(
                f"Warning: multiple files named {candidate.name}; "
                f"using {matches[0]}"
            )
        return matches[0]

    return None

# ============================================================
# SIGNAL EXTRACTION
# ============================================================

def sample_signal_using_distance(
    img,
    midpoint,
    target,
    slope,
    distance,
    real_world_peak_locations,
    real_world_peak_std,
    tolerance=30,
    track_width=15,
    show=False
):
    gray = img[:, :, 1]
    h, w = gray.shape

    angle = math.atan(slope)
    vx = math.cos(angle)
    vy = math.sin(angle)

    dx = target[0] - midpoint[0]
    dy = target[1] - midpoint[1]

    if vx * dx + vy * dy < 0:
        angle += math.pi
        vx = math.cos(angle)
        vy = math.sin(angle)

    x_start = midpoint[0] - tolerance * vx
    y_start = midpoint[1] - tolerance * vy
    x_end = midpoint[0] + (distance + tolerance) * vx
    y_end = midpoint[1] + (distance + tolerance) * vy

    num_steps = max(3, int(abs(distance) + 2 * tolerance))

    x_path = np.linspace(x_start, x_end, num_steps)
    y_path = np.linspace(y_start, y_end, num_steps)

    nx = -math.sin(angle)
    ny = math.cos(angle)

    offsets = np.arange(
        -(track_width // 2),
        track_width - track_width // 2
    )

    signal_sum = np.zeros(num_steps, dtype=float)

    for offset in offsets:
        sample_x = np.clip(
            (x_path + offset * nx).astype(int), 0, w - 1
        )
        sample_y = np.clip(
            (y_path + offset * ny).astype(int), 0, h - 1
        )

        signal_sum += gray[sample_y, sample_x]

    avg_signal = signal_sum / len(offsets)
    inverted = np.max(avg_signal) - avg_signal

    win = min(21, len(inverted) // 2 * 2 - 1)

    smooth = (
        savgol_filter(inverted, win, 2)
        if win >= 3 else inverted
    )

    baseline = gaussian_filter(smooth, sigma=20)
    final_signal = np.maximum(smooth - baseline, 0)

    if distance <= 0:
        raise ValueError("Calibration distance must be positive.")

    cm_per_pixel = 2.55 / distance

    # Detect the control peak on the sampled signal.
    peaks, _ = find_peaks(
        final_signal,
        prominence=max(float(np.max(final_signal)) * 0.05, 1e-8),
        width=4
    )

    if len(peaks) == 0:
        raise ValueError("No control peak detected.")

    control_idx = int(peaks[0])

    peak_locations = [control_idx]
    peak_intensities = [float(final_signal[control_idx])]

    for location_cm, std_cm in zip(
        real_world_peak_locations,
        real_world_peak_std
    ):
        expected_idx = (
            control_idx + location_cm / cm_per_pixel
        )

        std_px = std_cm / cm_per_pixel

        window_start = max(
            0, int(np.floor(expected_idx - 3 * std_px))
        )

        window_end = min(
            len(final_signal) - 1,
            int(np.ceil(expected_idx + 3 * std_px))
        )

        if window_start <= window_end:
            window = final_signal[window_start:window_end + 1]

            local_peaks, properties = find_peaks(
                window,
                prominence=max(float(np.max(window)) * 0.05, 1e-8),
                width=2
            )

            if len(local_peaks) > 0:
                # Choose the tallest detected local peak.
                local_idx = local_peaks[
                    np.argmax(window[local_peaks])
                ]

                peak_idx = window_start + int(local_idx)
                peak_intensity = float(final_signal[peak_idx])
            else:
                peak_idx = int(np.clip(
                    round(expected_idx),
                    0,
                    len(final_signal) - 1
                ))
                peak_intensity = 0.0
        else:
            peak_idx = int(np.clip(
                round(expected_idx),
                0,
                len(final_signal) - 1
            ))
            peak_intensity = 0.0

        peak_locations.append(peak_idx)
        peak_intensities.append(peak_intensity)

    if show:
        plt.figure(figsize=(12, 4))
        plt.plot(final_signal, label="Processed signal")
        plt.scatter(
            peak_locations,
            peak_intensities,
            color="red",
            label="Selected peaks"
        )
        plt.xlabel("Pixels along sampling path")
        plt.ylabel("Intensity (AU)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return (
        np.asarray(peak_locations, dtype=int),
        np.asarray(peak_intensities, dtype=float)
    )


# ============================================================
# PROCESS ONE PATIENT IMAGE
# ============================================================

def process_patient_image(
    image_path,
    image_name,
    condition="",
    notes=""
):
    result = {
        "Image": str(image_name),
        "Condition": condition,
        "Notes": notes,
        "Status": "ERROR",
        "Error": ""
    }

    try:
        img = cv2.imread(str(image_path))

        if img is None:
            raise ValueError(f"Could not read image: {image_path}")

        detections = yolo_model(str(image_path))

        if len(detections) == 0:
            raise ValueError("YOLO returned no detections.")

        box_c_locations, box_1_locations = (
            check_prediction_classes(detections[0])
        )

        midpoint, slope, distance, target = roi_slope_center(
            box_c_locations,
            box_1_locations
        )

        _, peak_intensities = sample_signal_using_distance(
            img=img,
            midpoint=midpoint,
            target=target,
            slope=slope,
            distance=distance,
            real_world_peak_locations=peak_locations_cm,
            real_world_peak_std=peak_std_cm,
            track_width=15,
            tolerance=30,
            show=False
        )

        if len(peak_intensities) < 7:
            raise ValueError(
                f"Expected 7 peak intensities; got {len(peak_intensities)}"
            )

        control = float(peak_intensities[0])

        result["Control_Intensity"] = control

        for band in range(1, 7):
            signal = float(peak_intensities[band])

            result[f"Band_{band}_Intensity"] = signal

            result[f"Band_{band}_Ratio"] = (
                signal / control if control != 0 else np.nan
            )

            # The model was trained on raw [Control, Signal],
            # not on the ratio alone.
            if not np.isfinite(control) or not np.isfinite(signal):
                raise ValueError(
                    f"Non-finite intensity for band {band}"
                )

            result[f"Band_{band}_Prediction"] = predict_band(
                control,
                signal,
                band_models[band]
            )

        # Diagnostic plot
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[0].set_title(str(image_name))
        axes[0].axis("off")

        axes[1].bar(
            ["Control"] + [f"Band {i}" for i in range(1, 7)],
            peak_intensities[:7]
        )
        axes[1].set_ylabel("Processed intensity (AU)")
        axes[1].tick_params(axis="x", rotation=45)

        fig.tight_layout()

        plot_path = PLOT_DIR / (
            f"{Path(image_name).stem}_diagnostic.png"
        )

        fig.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        result["Diagnostic_Plot"] = str(plot_path)
        result["Status"] = "OK"

    except Exception as error:
        result["Error"] = (
            f"{type(error).__name__}: {error}"
        )

    return result


# ============================================================
# INITIALIZE YOLO AND CALIBRATION
# ============================================================

if not PATIENT_CSV.is_file():
    raise FileNotFoundError(f"Patient CSV not found: {PATIENT_CSV}")

if not YOLO_PATH.is_file():
    raise FileNotFoundError(f"YOLO model not found: {YOLO_PATH}")

# These helper functions must be defined or imported:
# check_prediction_classes()
# roi_slope_center()

# Calibration arrays must already be saved or reconstructed
# from your calibration measurements.
#
# Expected variables:
# band_1_mean, band_1_std, ..., band_6_mean, band_6_std
#
# They must use the same units as the distance calibration.

required_calibration = [
    f"band_{i}_{stat}"
    for i in range(1, 7)
    for stat in ("mean", "std")
]

missing_calibration = [
    name for name in required_calibration
    if name not in globals()
]

if missing_calibration:
    raise RuntimeError(
        "Missing calibration variables: "
        + ", ".join(missing_calibration)
        + ". Define or load these values before processing images."
    )

peak_locations_cm = [
    globals()[f"band_{i}_mean"]
    for i in range(1, 7)
]

peak_std_cm = [
    globals()[f"band_{i}_std"]
    for i in range(1, 7)
]

yolo_model = YOLO(str(YOLO_PATH))


# ============================================================
# READ CSV AND PROCESS ALL PATIENTS
# ============================================================

patients = pd.read_csv(PATIENT_CSV)
patients['Directory'] = 'patient_samples'

if "Image" not in patients.columns:
    raise ValueError(
        f"Expected 'Image' column. Found {patients.columns.tolist()}"
    )

if "Condition" not in patients.columns:
    patients["Condition"] = ""

if "Unnamed: 2" in patients.columns:
    patients["Notes"] = patients["Unnamed: 2"].fillna("")
elif "Notes" not in patients.columns:
    patients["Notes"] = ""

all_results = []

for index, patient in patients.iterrows():
    image_name = str(patient["Image"]).strip()
    condition = patient.get("Condition", "")
    notes = patient.get("Notes", "")

    image_path = find_image(image_name)

    print(
        f"[{index + 1}/{len(patients)}] "
        f"Processing {image_name}..."
    )

    if image_path is None:
        result = {
            "Image": image_name,
            "Condition": condition,
            "Notes": notes,
            "Status": "ERROR",
            "Error": f"Image not found: {image_name}"
        }
    else:
        result = process_patient_image(
            image_path=image_path,
            image_name=image_name,
            condition=condition,
            notes=notes
        )

    all_results.append(result)
    print(f"  Status: {result['Status']}")

    if result["Status"] == "ERROR":
        print(f"  Error: {result['Error']}")


# ============================================================
# SAVE AND DOWNLOAD RESULTS
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = OUTPUT_DIR / "patient_predictions.csv"
results_df.to_csv(results_path, index=False)

print("\nProcessing complete.")
print("Results saved to:", results_path)
print(results_df["Status"].value_counts(dropna=False))

display(results_df.head())

from IPython.display import display
from google.colab import files

files.download(str(results_path))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load inference results
results_path = Path("/content/patient_results/patient_predictions.csv")
df = pd.read_csv(results_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nProcessing status:")
print(df["Status"].value_counts(dropna=False))

print("\nCondition counts:")
print(df["Condition"].value_counts(dropna=False))

display(df.head())


# Separate successful and failed cases
successful = df[df["Status"] == "OK"].copy()
failed = df[df["Status"] != "OK"].copy()

print(f"Total images: {len(df)}")
print(f"Successfully processed: {len(successful)}")
print(f"Failed: {len(failed)}")

if not failed.empty:
    display(failed[["Image", "Condition", "Error"]])

# Check missing values in prediction columns
prediction_cols = [
    f"Band_{i}_Prediction" for i in range(1, 7)
]

print("\nMissing predictions:")
display(successful[prediction_cols].isna().sum().to_frame("Missing"))

# Descriptive statistics
display(successful[prediction_cols].describe().T)

In [ ]:
correlation = successful[prediction_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)
plt.title("Correlation Between Band Predictions")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))

sns.violinplot(
    data=plot_data,
    x="Condition",
    y="Prediction",
    hue="Band",
    inner="quartile",
    cut=0,
    density_norm="width"
)

plt.title("Distribution of Band Predictions by Patient Condition")
plt.xlabel("Patient Condition")
plt.ylabel("Predicted Value")
plt.xticks(rotation=45, ha="right")
plt.legend(
    title="Band",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:

# ============================================================
# 5-FOLD STRATIFIED CV + HELD-OUT EMBEDDING UMAP
# ============================================================

# In Colab, install if needed:
# %pip install -q umap-learn

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import umap

# ------------------------------------------------------------
# 1. CONFIGURATION AND DATA
# ------------------------------------------------------------

CSV_PATH = Path("/content/patient_results/patient_predictions.csv")
OUTPUT_DIR = Path("/content/patient_results/cv5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5
EPOCHS = 200
LEARNING_RATE = 0.001

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

feature_cols = [
    f"Band_{i}_Prediction" for i in range(1, 7)
]

df = pd.read_csv(CSV_PATH)

data = df.loc[df["Status"].eq("OK")].copy()
data["Condition"] = data["Condition"].astype("string").str.strip()

data[feature_cols] = data[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=feature_cols + ["Condition"])
data = data[data["Condition"] != ""]

X = data[feature_cols].to_numpy(dtype=np.float32)

encoder = LabelEncoder()
y = encoder.fit_transform(data["Condition"].astype(str))

classes = encoder.classes_
n_classes = len(classes)

class_counts = np.bincount(y)

if class_counts.min() < N_SPLITS:
    raise ValueError(
        f"5-fold stratified CV requires at least 5 samples "
        f"in every class. Counts: "
        f"{dict(zip(classes, class_counts))}"
    )

print("Samples:", len(X))
print("Classes:", dict(zip(classes, class_counts)))
print("Device:", DEVICE)


# ------------------------------------------------------------
# 2. CLASSIFIER
# ------------------------------------------------------------

class ConditionClassifier(nn.Module):
    def __init__(self, input_size=6, num_classes=2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        return self.network(x)


# ------------------------------------------------------------
# 3. 5-FOLD CROSS-VALIDATION
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_predictions = np.full(len(y), -1, dtype=int)
oof_probabilities = np.zeros((len(y), n_classes))
oof_embeddings = np.full((len(y), 16), np.nan)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):

    print(f"\n{'=' * 20} FOLD {fold}/{N_SPLITS} {'=' * 20}")

    # Split raw features
    X_train_raw, X_test_raw = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit preprocessing ONLY on the training fold
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_test = scaler.transform(X_test_raw).astype(np.float32)

    X_train_tensor = torch.tensor(
        X_train, dtype=torch.float32, device=DEVICE
    )
    X_test_tensor = torch.tensor(
        X_test, dtype=torch.float32, device=DEVICE
    )
    y_train_tensor = torch.tensor(
        y_train, dtype=torch.long, device=DEVICE
    )

    # Reset model initialization for each fold
    torch.manual_seed(RANDOM_STATE + fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE + fold)

    model = ConditionClassifier(
        input_size=6,
        num_classes=n_classes
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )

    # Train using only the training fold
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()

        logits = model(X_train_tensor)
        loss = criterion(logits, y_train_tensor)

        loss.backward()
        optimizer.step()

    # Evaluate on the held-out fold
    model.eval()

    with torch.no_grad():
        logits = model(X_test_tensor)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1).cpu().numpy()

        # Penultimate layer: 16-dimensional embeddings
        embeddings = (
            model.network[:-1](X_test_tensor)
            .cpu()
            .numpy()
        )

    oof_predictions[test_idx] = predictions
    oof_probabilities[test_idx] = probabilities.cpu().numpy()
    oof_embeddings[test_idx] = embeddings

    fold_acc = accuracy_score(y_test, predictions)
    fold_bal_acc = balanced_accuracy_score(y_test, predictions)
    fold_f1 = f1_score(
        y_test, predictions,
        average="macro", zero_division=0
    )

    fold_results.append({
        "Fold": fold,
        "Accuracy": fold_acc,
        "Balanced_Accuracy": fold_bal_acc,
        "Macro_F1": fold_f1
    })

    print(f"Accuracy:          {fold_acc:.4f}")
    print(f"Balanced accuracy: {fold_bal_acc:.4f}")
    print(f"Macro F1:          {fold_f1:.4f}")


# ------------------------------------------------------------
# 4. SUMMARIZE CROSS-VALIDATION
# ------------------------------------------------------------

fold_df = pd.DataFrame(fold_results)

print("\nPer-fold metrics:")
print(fold_df.round(4).to_string(index=False))

print("\nMean and standard deviation across folds:")
print(
    fold_df.drop(columns="Fold")
    .agg(["mean", "std"])
    .round(4)
)

# Aggregate out-of-fold predictions
print("\nOverall out-of-fold classification report:")
print(
    classification_report(
        y,
        oof_predictions,
        labels=np.arange(n_classes),
        target_names=classes,
        zero_division=0
    )
)

# Save fold metrics and predictions
fold_df.to_csv(
    OUTPUT_DIR / "fold_metrics.csv", index=False
)

prediction_df = data[["Image", "Condition"]].copy()
prediction_df["Actual_Class"] = encoder.inverse_transform(y)
prediction_df["Predicted_Class"] = encoder.inverse_transform(
    oof_predictions
)
prediction_df["Correct"] = (
    prediction_df["Actual_Class"]
    == prediction_df["Predicted_Class"]
)

for i, class_name in enumerate(classes):
    prediction_df[f"Probability_{class_name}"] = (
        oof_probabilities[:, i]
    )

prediction_df.to_csv(
    OUTPUT_DIR / "oof_predictions.csv", index=False
)


# ------------------------------------------------------------
# 5. OUT-OF-FOLD CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y,
    oof_predictions,
    labels=np.arange(n_classes)
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted Condition")
plt.ylabel("Actual Condition")
plt.title("5-Fold CV: Out-of-Fold Confusion Matrix")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "oof_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("\nAll outputs saved to:", OUTPUT_DIR)


In [ ]:

# ============================================================
# IDENTIFY MISCLASSIFIED IMAGES FROM 5-FOLD CV
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Original sample indices correspond to the OOF predictions
error_df = data[["Image", "Condition"]].copy().reset_index(drop=True)

error_df["Actual_Class"] = encoder.inverse_transform(y)
error_df["Predicted_Class"] = encoder.inverse_transform(
    oof_predictions
)

error_df["Correct"] = (
    error_df["Actual_Class"] == error_df["Predicted_Class"]
)

# Add the six predicted concentrations for investigation
for col in feature_cols:
    error_df[col] = data[col].to_numpy()

# Keep only incorrect predictions
misclassified = (
    error_df.loc[~error_df["Correct"]]
    .sort_values(["Actual_Class", "Predicted_Class", "Image"])
    .reset_index(drop=True)
)

print(f"Total images: {len(error_df)}")
print(f"Correct: {error_df['Correct'].sum()}")
print(f"Incorrect: {len(misclassified)}")
print(f"Error rate: {(~error_df['Correct']).mean():.2%}")

display(misclassified)

# Save error list
error_path = Path(OUTPUT_DIR) / "misclassified_images.csv"
misclassified.to_csv(error_path, index=False)

print("Misclassified images saved to:", error_path)


In [ ]:

# ============================================================
# VISUALIZE MISCLASSIFIED PATIENT IMAGES
# ============================================================

import math
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

IMAGE_DIR = Path("/content/patient_samples")

def locate_image(filename):
    """Search only the specified directory, not its subdirectories."""
    path = IMAGE_DIR / str(filename).strip()
    return path if path.is_file() else None

if misclassified.empty:
    print("No misclassified images in the out-of-fold predictions.")
else:
    n_images = len(misclassified)
    ncols = 3
    nrows = math.ceil(n_images / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(5 * ncols, 4.5 * nrows),
        squeeze=False
    )

    for ax, (_, row) in zip(axes.flat, misclassified.iterrows()):
        filename = row["Image"]
        image_path = locate_image(filename)

        if image_path is None:
            ax.text(
                0.5, 0.5,
                f"Image not found\n{filename}",
                ha="center", va="center"
            )
            ax.axis("off")
            continue

        image = cv2.imread(str(image_path))

        if image is None:
            ax.text(0.5, 0.5, "Could not read image", ha="center")
            ax.axis("off")
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.set_title(
            f"{filename}\n"
            f"Actual: {row['Actual_Class']}\n"
            f"Predicted: {row['Predicted_Class']}",
            fontsize=10,
            color="red"
        )
        ax.axis("off")

    for ax in axes.flat[n_images:]:
        ax.axis("off")

    plt.suptitle("Misclassified Images — 5-Fold Cross-Validation")
    plt.tight_layout()
    plt.show()
